In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo
from src.simulation import predict_match, assign_thirds
from src.features import update_team_history, update_h2h
import matplotlib as plt
import numpy as np

In [2]:
#Load everything from earlier notebooks
draw_threshold = joblib.load('draw_threshold.joblib')
wc_model = joblib.load('world_cup_model.joblib')
history_dict = joblib.load('final_history.joblib')
h2h_dict = joblib.load('final_h2h.joblib')
country_elo = joblib.load('final_elo.joblib')
model_h = joblib.load('home_goals_model.joblib')
model_a = joblib.load('away_goals_model.joblib')
scaler = joblib.load('scaler.joblib')

#Make a dataframe containing all the games in the group stage
group_stage_matches = load_matches()
group_stage_matches = prepare_matches(group_stage_matches, "2026-06-10", "2026-06-28")

In [3]:
#Group stage simulation

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

rows=[]
for group, teams in wc_groups.items():
    for team in teams:
        rows.append({"Group": group,
                     "Team": team,
                     "Points": 0,
                     "GD": 0,
                     'GF': 0,
                     "GA": 0})
        
group_stage_result = pd.DataFrame(rows) #Gives us the starting group stage
for match in group_stage_matches.itertuples(index = False):
    
    x = predict_match(match.home_team, match.away_team, wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler)
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'Points'] += x.home_points
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'Points'] += x.away_points
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GD'] += x.home_score - x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GD'] += x.away_score - x.home_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GF'] += x.home_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GF'] += x.away_score
    
    group_stage_result.loc[group_stage_result['Team'] == match.home_team, 'GA'] += x.away_score
    group_stage_result.loc[group_stage_result['Team'] == match.away_team, 'GA'] += x.home_score
    
    if x.outcome == "Home Win": S = 1
    elif x.outcome == "Away Win": S = 0
    else: S = 0.5

    new_away, new_home = update_elo(S, match.neutral, match.K_factor, match.home_score, match.away_score, 
                                    country_elo[match.away_team], 
                                    country_elo[match.home_team])
    
    country_elo[match.home_team] = new_home
    country_elo[match.away_team] = new_away
    
    update_team_history(match, history_dict)
    update_h2h(match, h2h_dict)

group_stage_result = group_stage_result.sort_values(['Group', 'Points', 'GD', 'GF'], ascending=[True, False, False, False]).reset_index(drop=True)    
print(group_stage_result) #Predicted group stage results


   Group                    Team  Points  GD  GF  GA
0      A                  Mexico       9   6   7   1
1      A          Czech Republic       4  -2   3   5
2      A            South Africa       3  -2   1   3
3      A             South Korea       1  -2   0   2
4      B                  Canada       9   4   5   1
5      B             Switzerland       6   4   5   1
6      B  Bosnia and Herzegovina       3  -5   4   9
7      B                   Qatar       0  -3   2   5
8      C                Scotland       9   3   8   5
9      C                 Morocco       4   2   6   4
10     C                  Brazil       4   0   2   2
11     C                   Haiti       0  -5   3   8
12     D           United States       5   1   2   1
13     D                  Turkey       4   1   4   3
14     D                Paraguay       4  -1   3   4
15     D               Australia       2  -1   1   2
16     E                 Ecuador       7   5   8   3
17     E                 Germany       4  -2  

In [11]:
#Get teams that move on to round of 32

top2 = group_stage_result.groupby('Group').head(2).copy() #Teams that placed top 2 in their group which qualifies
third = group_stage_result.groupby('Group').nth(2).copy() #All teams that placed third (only 8 of them move on)

best8_third = third.sort_values(
    ['Points', 'GD', 'GF'], 
    ascending=[False, False, False]
).head(8)

thirds_slot_map = {
    'E': 'ABCDF',
    'I': 'CDFGH', 
    'A': 'CEFHI',
    'L': 'EHIJK',
    'D': 'BEFIJ',
    'G': 'AEHIJ',
    'B': 'EFGIJ',
    'K': 'DEIJL',
}

winners = {g: teams.iloc[0]['Team'] for g, teams in group_stage_result.groupby('Group')}
runners = {g: teams.iloc[1]['Team'] for g, teams in group_stage_result.groupby('Group')}

r32_pairings = [
    (runners['A'], runners['B']),
    (winners['F'], runners['C']),
    (winners['C'], runners['F']),
    (runners['E'], runners['I']),
    (winners['H'], runners['J']),
    (winners['J'], runners['H']),
    (runners['K'], runners['L']),
    (runners['D'], runners['G']),
]

available_thirds = {row.Group: row.Team for row in best8_third.itertuples()}
assignments = {}
used = set()

for winner_group, eligible_groups in thirds_slot_map.items():
    for g in eligible_groups:
        if g in available_thirds and g not in used:
            assignments[winner_group] = available_thirds[g]
            used.add(g)
            break

third_pairings = [(winners[g], third_team) for g, third_team in assignments.items()]

r32_rows = []
r16_teams = []

for home, away in r32_pairings + third_pairings:
    x=predict_match(home, away, wc_model, draw_threshold, history_dict, h2h_dict, country_elo, model_h, model_a, scaler)

    if x.outcome == "Home Win": 
        S, winner = 1, home
    elif x.outcome == "Away Win": 
        S, winner = 0, away
    else: 
        if x.diff > 0:
            S, winner = 1, home
            x.home_score += 1
        else:
            S, winner = 0, away
            x.away_score += 1

    r16_teams.append(winner)
    r32_rows.append({
        'home_team': home,
        'away_team': away,
        'home_score': x.home_score,
        'away_score': x.away_score,
        'result': S,
        'neutral': 1,
        'K_factor': 50,
        'winner': winner
    })
    
    print(f"{home} {x.home_score} - {x.away_score} {away} → {winner}")

    new_away, new_home = update_elo(S, 1, 50, x.home_score, x.away_score, 
                                    country_elo[match.away_team], 
                                    country_elo[match.home_team])
    
    country_elo[match.home_team] = new_home
    country_elo[match.away_team] = new_away
    
    
    
    
    


Czech Republic 2 - 1 Switzerland → Czech Republic
Netherlands 1 - 0 Morocco → Netherlands
Scotland 1 - 0 Japan → Scotland
Germany 5 - 0 France → Germany
Cape Verde 1 - 0 Argentina → Cape Verde
Jordan 1 - 0 Uruguay → Jordan
DR Congo 2 - 1 England → DR Congo
Turkey 4 - 3 Egypt → Turkey
Ecuador 1 - 0 Brazil → Ecuador
Norway 1 - 0 Paraguay → Norway
Mexico 2 - 0 Ivory Coast → Mexico
Croatia 1 - 0 Spain → Croatia
United States 2 - 1 Tunisia → United States
Belgium 2 - 1 Senegal → Belgium
Canada 2 - 1 Algeria → Canada


Czech Republic 1 - 0 Switzerland -> Winner: Czech Republic


AttributeError: 'Pandas' object has no attribute 'result'